In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE 
from sklearn.preprocessing import StandardScaler
import sys
import io
import warnings
import time

# --- Global Settings ---
warnings.filterwarnings('ignore')
start_time = time.time()
audit_log = []

# 1. SET YOUR FILENAME (Mock Data Generation)
file_name = 'synthetic_hotspot_data.xlsx' 

audit_log.append("="*60)
audit_log.append(" 3.0 Advanced Modeling: Logistic Regression (Hotspot Prediction)")
audit_log.append("="*60)

# --- 1. Load and Generate Data ---
log_entry = f"\n--- 1. Data Generation and Loading ---\n"
print(log_entry)
audit_log.append(log_entry)

N_TRAIN = 500 
N_TEST = 100
np.random.seed(42)

def generate_features(n):
    return pd.DataFrame({
        'Road_Condition': np.random.randint(1, 6, n),
        'Weather_Type': np.random.randint(1, 10, n),
        'Speed_Limit': np.random.randint(30, 100, n),
        'Time_of_Day_Encoded': np.random.rand(n) * 24,
        'Population_Density': np.random.lognormal(mean=2, sigma=0.5, size=n),
        'Lat_Grid': 30 + np.random.rand(n) * 5, 
        'Lng_Grid': 40 + np.random.rand(n) * 5, 
    })

def generate_severity(df):
    p = 1 / (1 + np.exp(-(
        1.2 * df['Road_Condition'] +
        0.5 * df['Weather_Type'] -
        0.08 * df['Speed_Limit'] +
        0.2 * df['Population_Density'] +
        (0.8 * df['Road_Condition'] * df['Population_Density'] / 10) +
        np.random.normal(0, 4, len(df))
    ) - 6))
    return (p > 0.5).astype(int)

train_features = generate_features(N_TRAIN)
test_features = generate_features(N_TEST)

train_df = train_features.copy()
train_df['Severity'] = generate_severity(train_features)
test_df = test_features.copy()
test_df['Severity'] = generate_severity(test_features)

if train_df['Severity'].sum() == 0:
    train_df.loc[np.random.choice(train_df.index, size=int(N_TRAIN*0.1), replace=False), 'Severity'] = 1

log_entry = f"Synthetic Training Data Size: {train_df.shape[0]}. Hotspot Count (1): {train_df['Severity'].sum()}"
print(log_entry)
audit_log.append(log_entry)


# --- 2. Data Preparation ---
TARGET_COLUMN = 'Severity' 
NON_FEATURE_COLUMNS = ['Lat_Grid', 'Lng_Grid']

X_train_raw = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]
X_test_raw = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]

X_test_coords = X_test_raw[NON_FEATURE_COLUMNS].reset_index(drop=True)


# --- 3. Feature Engineering and Scaling (Advanced Preprocessing) ---
log_entry = "\n--- 3. Feature Engineering and Scaling ---\n"
print(log_entry)
audit_log.append(log_entry)

# 1. Feature Interaction Term
X_train_raw['Road_Density_Interaction'] = X_train_raw['Road_Condition'] * X_train_raw['Population_Density']
X_test_raw['Road_Density_Interaction'] = X_test_raw['Road_Condition'] * X_test_raw['Population_Density']

X_train_features = X_train_raw.drop(columns=NON_FEATURE_COLUMNS, errors='ignore')
X_test_features = X_test_raw.drop(columns=NON_FEATURE_COLUMNS, errors='ignore')

# 2. Data Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)

feature_names = X_train_features.columns
X_train = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test = pd.DataFrame(X_test_scaled, columns=feature_names)

log_entry = f"Features used for modeling: {list(feature_names)}"
print(log_entry)
audit_log.append(log_entry)


# --- 4. Handling Imbalance (SMOTE) ---
log_entry = "\n--- 4. Handling Imbalance (SMOTE) ---\n"
log_entry += f"Original Hotspot Count: {y_train.sum()}"
print(log_entry)
audit_log.append(log_entry)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

log_entry = f"SMOTE Hotspot Count: {y_train_smote.sum()}"
print(log_entry)
audit_log.append(log_entry)


# --- 5. Train the Model ---
log_entry = "\n--- 5. Training Logistic Regression Model ---\n"
print(log_entry)
audit_log.append(log_entry)

log_model = LogisticRegression(
    max_iter=1000, 
    random_state=42,
    solver='liblinear',
    class_weight='balanced'
)
log_model.fit(X_train_smote, y_train_smote) 

log_entry = "Model training complete."
print(log_entry)
audit_log.append(log_entry)


# --- 6. Make Predictions ---
y_pred = log_model.predict(X_test)
y_pred_proba = log_model.predict_proba(X_test)[:, 1]


# --- 7. Specialized Evaluation and Interpretation ---
log_entry = "\n--- 7. Specialized Evaluation and Interpretation ---\n"
print(log_entry)
audit_log.append(log_entry)

# Metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)

# Odds Ratios
odds_ratios = np.exp(log_model.coef_[0])
odds_ratios_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': log_model.coef_[0],
    'Odds_Ratio': odds_ratios
}).sort_values(by='Odds_Ratio', ascending=False)
odds_ratios_string = odds_ratios_df.to_string(index=False)

old_stdout = sys.stdout
sys.stdout = classification_report_output = io.StringIO()
print(classification_report(y_test, y_pred, zero_division=0))
sys.stdout = old_stdout
report_string = classification_report_output.getvalue()

conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix_string = str(conf_matrix)

log_entry = f"ROC-AUC: {roc_auc:.4f}, Accuracy: {accuracy:.4f}"
print(log_entry)
audit_log.append(log_entry)


# --- 8. Create Plotting Dataframe ---
plot_df = X_test.copy()
plot_df['Lat_Grid'] = X_test_coords['Lat_Grid'].values
plot_df['Lng_Grid'] = X_test_coords['Lng_Grid'].values
plot_df['Actual'] = y_test.values
plot_df['Predicted'] = y_pred
plot_df['Predicted_Proba'] = y_pred_proba


### 9. ADVANCED VISUALIZATIONS ###
log_entry = "\n--- 9. Generating Advanced Visualizations ---\n"
print(log_entry)
audit_log.append(log_entry)

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=plot_df,
    x='Lng_Grid', y='Lat_Grid',
    hue='Predicted', 
    style='Actual', 
    palette={0: 'gray', 1: 'red'},
    markers={0: 'o', 1: 'X'},
    s=150, alpha=0.7
)
plt.title('Predicted Hotspots vs Actual Locations (Test Data)')
plt.xlabel('Longitude Grid (Lng_Grid)')
plt.ylabel('Latitude Grid (Lat_Grid)')
plt.legend(title='Predicted / Actual')
plt.tight_layout()
plt.savefig('predicted_hotspot_map_LR.png')
audit_log.append("Saved 'predicted_hotspot_map_LR.png'")
plt.close()

cm = confusion_matrix(plot_df['Actual'], plot_df['Predicted'])
class_labels = ['Not-Hotspot (0)', 'Is-Hotspot (1)']
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('confusion_matrix_LR.png')
audit_log.append("Saved 'confusion_matrix_LR.png'")
plt.close()

plt.figure(figsize=(10, 6))
sns.barplot(x='Odds_Ratio', y='Feature', data=odds_ratios_df)
plt.axvline(x=1, color='r', linestyle='--', label='Neutral Effect (Odds Ratio = 1)')
plt.title('Feature Impact on Hotspot Likelihood (Odds Ratio)')
plt.xlabel('Odds Ratio (Effect on Hotspot Probability)')
plt.ylabel('Feature')
plt.legend()
plt.tight_layout()
plt.savefig('odds_ratio_impact_LR.png')
audit_log.append("Saved 'odds_ratio_impact_LR.png'")
plt.close()


# --- 10. Final Report and Audit File ---
log_entry = "\n--- 10. Generating Final Report ---\n"
print(log_entry)
audit_log.append(log_entry)

end_time = time.time()
total_time = end_time - start_time

results_content = f"""
==================================================
Advanced Modeling: Logistic Regression Results
==================================================

1. Accuracy Score (Overall): {accuracy:.4f}

2. ROC-AUC Score (Crucial for Hotspot Prediction): {roc_auc:.4f}

3. Classification Report (Focus on Hotspot (1) Recall/Precision):
{report_string.strip()}

4. Confusion Matrix (True vs. Predicted):
{conf_matrix_string}

5. Feature Interpretation (Odds Ratios):
> Interpretation: An Odds Ratio > 1 means the feature increases the probability of being a Hotspot.
{odds_ratios_string}

6. Model Parameters:
{str(log_model)}

7. Visualizations Saved:
- predicted_hotspot_map_LR.png
- confusion_matrix_LR.png
- odds_ratio_impact_LR.png

==================================================
Total execution time: {total_time:.2f} seconds.
"""

output_filename = 'advanced_logistic_regression_results_final.txt'
with open(output_filename, 'w') as f:
    f.write(results_content.strip())

audit_file_name = "LR_model_audit_report.txt"
audit_content = "\n".join(audit_log)
with open(audit_file_name, "w", encoding="utf-8") as f:
    f.write(audit_content)

print(f"Successfully saved detailed report to {output_filename}.")
print(f"Successfully saved audit log to {audit_file_name}.")
print("\n--- Final Results Summary ---\n")
print(results_content.strip())
print("\nAll tasks complete.")


--- 1. Data Generation and Loading ---

Synthetic Training Data Size: 500. Hotspot Count (1): 492

--- 3. Feature Engineering and Scaling ---

Features used for modeling: ['Road_Condition', 'Weather_Type', 'Speed_Limit', 'Time_of_Day_Encoded', 'Population_Density', 'Road_Density_Interaction']

--- 4. Handling Imbalance (SMOTE) ---
Original Hotspot Count: 492
SMOTE Hotspot Count: 492

--- 5. Training Logistic Regression Model ---

Model training complete.

--- 7. Specialized Evaluation and Interpretation ---

ROC-AUC: 0.7552, Accuracy: 0.8600

--- 9. Generating Advanced Visualizations ---


--- 10. Generating Final Report ---

Successfully saved detailed report to advanced_logistic_regression_results_final.txt.
Successfully saved audit log to LR_model_audit_report.txt.

--- Final Results Summary ---

Advanced Modeling: Logistic Regression Results

1. Accuracy Score (Overall): 0.8600

2. ROC-AUC Score (Crucial for Hotspot Prediction): 0.7552

3. Classification Report (Focus on Hotspot (